# Kalliope SRU Abfrage und parsen von MODS mit Python etree
Sources: Code f+r die SRU-Abfrage und das Parsen der Daten adaptiert nach DNB SRU Tutorial

https://github.com/deutsche-nationalbibliothek/dnblab/blob/main/DNB_SRU_Tutorial.ipynb 


In [4]:
import requests
from lxml import etree
import pandas as pd

In [8]:
# SRU query
def kalliope_sru(query):
    base_url = "https://kalliope-verbund.info/sru"
    params = {
        'version': '1.2',
        'operation': 'searchRetrieve',
        'recordSchema': 'mods37',
        'maximumRecords': '1000',
        #mehr records werden gefunden, wenn maximum records hochgesetzt wird.
        'query': query
    }
    
    r = requests.get(base_url, params=params)
    mods_content = r.content
    records_mods = etree.fromstring(mods_content)
    
    # Check if more than 100 records
    if len(records_mods.xpath("//srw:record", namespaces={'srw': 'http://www.loc.gov/zing/srw/'})) < 100:
        return records_mods
    else:
        num_results = 100
        i = 101
        while num_results == 100:
            params.update({'startRecord': i})
            r = requests.get(base_url, params=params)
            new_mods_content = r.content
            new_records_mods = etree.fromstring(new_mods_content)
            records_mods.extend(new_records_mods.xpath("//srw:record", namespaces={'srw': 'http://www.loc.gov/zing/srw/'}))
            i += 100
            num_results = len(new_records_mods.xpath("//srw:record", namespaces={'srw': 'http://www.loc.gov/zing/srw/'}))
        
        return records_mods


In [9]:
def parse_mods(record):
    ns = {
        'srw': 'http://www.loc.gov/zing/srw/',  # SRW namespace
        'mods': 'http://www.loc.gov/mods/v3'    # MODS namespace
    }
    
    # Extract RecordID
    recordIdentifier = record.xpath(".//mods:mods/mods:recordInfo/mods:recordIdentifier", namespaces=ns)
    recordIdentifier = recordIdentifier[0].text if recordIdentifier else "unknown"
    
    # Extract Title
    title = record.xpath(".//mods:mods/mods:titleInfo/mods:title", namespaces=ns)
    title = title[0].text if title else "unknown"
    
    # Extract Date
    date = record.xpath(".//mods:mods/mods:originInfo/mods:dateCreated", namespaces=ns)
    date = date[0].text if date else "unknown"
    
    # Extract Names and Roles
    names = record.xpath(".//mods:mods/mods:name", namespaces=ns)
    senders = []
    receivers = []
    mentioned = []
    
    for name in names:
        name_text = name.xpath(".//mods:namePart/text()", namespaces=ns)
        role_text = name.xpath(".//mods:role/mods:roleTerm[@type='text']/text()", namespaces=ns)
        
        if name_text and role_text:
            name_value = name_text[0]
            role_value = role_text[0].lower()
            
            # Categorize based on role
            if "verfasser" in role_value:  # Adjust to match actual role values
                senders.append(name_value)
            elif "adressat" in role_value:
                receivers.append(name_value)
            elif "erwähnt" in role_value:
                mentioned.append(name_value)
    
    # Extract Genre
    genre_letter = record.xpath(".//mods:mods/mods:genre[text()='Brief']", namespaces=ns)
    genre_letter = genre_letter[0].text if genre_letter else "unknown"
    
    # Return a dictionary to build the DataFrame
    return {
        "recordIdentifier": recordIdentifier,
        "title": title,
        "date": date,
        "senders": "; ".join(senders),    # Combine names into a single string
        "receivers": "; ".join(receivers),
        "mentioned": "; ".join(mentioned),
        "genre": genre_letter
    }


In [10]:
# Example query

query = 'ead.archdesc.id="DE-611-BF-73161"'

#'ead.archdesc.id'
#query = "ead.addressee"=="Heisenberg"
records_xml = kalliope_sru(query)

print(f'{len(records_xml.xpath("//srw:record", namespaces={"srw": "http://www.loc.gov/zing/srw/"}))} Ergebnisse gefunden')


2000 Ergebnisse gefunden


In [11]:
# Parse data and convert to DataFrame
records = records_xml.xpath("//srw:record", namespaces={"srw": "http://www.loc.gov/zing/srw/"})
output = [parse_mods(record) for record in records]
df = pd.DataFrame(output)
df


,recordIdentifier,title,date,senders,receivers,mentioned,genre
0,DE-611-HS-3616988,"Brief von Werner Heisenberg an Václav Votruba,...",1969-01-14,"Heisenberg, Werner (1901-1976)","Votruba, Václav (1909-1990)","Heisenberg, Elisabeth (1914-1998)",Brief
1,DE-611-HS-3616991,"Brief von Václav Votruba an Werner Heisenberg,...",1968-12-31,"Votruba, Václav (1909-1990)","Heisenberg, Werner (1901-1976)",,Brief
2,DE-611-BF-81594,"487. IV. Institutionen, 1. Korrespondenz: Vaid...",1955,,,,unknown
3,DE-611-HS-3686363,Brief von Werner Heisenberg an J. V. Kotwal an...,1962-09-18,"Heisenberg, Werner (1901-1976)","Kotwal, J. V.; Tata Institute of Fundamental R...","Vaidya, Shreeniwas Maheshwar (1934-); Symanzik...",Brief
4,DE-611-HS-3686373,Brief von J. V. Kotwal von Tata Institute of F...,1962-09-12,"Kotwal, J. V.; Tata Institute of Fundamental R...","Heisenberg, Werner (1901-1976)","Vaidya, Shreeniwas Maheshwar (1934-)",Brief
...,...,...,...,...,...,...,...
1995,DE-611-HS-3626349,Brief von Annemarie Giese von Max-Planck-Insti...,1966,"Giese, Annemarie; Max-Planck-Institut für Phys...",Zentralstelle für Arbeitsvermittlung (-2007),"Hermann, Klaus-Otto",Brief
1996,DE-611-HS-3626351,Brief von Zentralstelle für Arbeitsvermittlung...,1966-03-18,Zentralstelle für Arbeitsvermittlung (-2007),"Heisenberg, Werner (1901-1976); Max-Planck-Ins...","Hermann, Klaus-Otto",Brief
1997,DE-611-HS-3626361,Brief von Zentralstelle für Atomkernenergie-Do...,1968-08-14,Zentralstelle für Atomkernenergie-Dokumentatio...,"Heisenberg, Werner (1901-1976); Max-Planck-Ins...",,Brief
1998,DE-611-HS-3626366,Brief von Zentralstelle für Atomkernenergie-Do...,1968-03-25,Zentralstelle für Atomkernenergie-Dokumentatio...,"Heisenberg, Werner (1901-1976); Max-Planck-Ins...",,Brief


In [13]:
#print(etree.tostring(records_xml, pretty_print=True).decode())

In [14]:
df.to_csv("heisenberg_namessplit.csv", index=False)

In [15]:
#df_bibsonomy_Europa_publications = df_bibsonomy_Europa.loc[df_bibsonomy_Europa['type'] == 'Publication']
df_B = df.loc[df["genre"] == "Brief"]
df_B

,recordIdentifier,title,date,senders,receivers,mentioned,genre
0,DE-611-HS-3616988,"Brief von Werner Heisenberg an Václav Votruba,...",1969-01-14,"Heisenberg, Werner (1901-1976)","Votruba, Václav (1909-1990)","Heisenberg, Elisabeth (1914-1998)",Brief
1,DE-611-HS-3616991,"Brief von Václav Votruba an Werner Heisenberg,...",1968-12-31,"Votruba, Václav (1909-1990)","Heisenberg, Werner (1901-1976)",,Brief
3,DE-611-HS-3686363,Brief von Werner Heisenberg an J. V. Kotwal an...,1962-09-18,"Heisenberg, Werner (1901-1976)","Kotwal, J. V.; Tata Institute of Fundamental R...","Vaidya, Shreeniwas Maheshwar (1934-); Symanzik...",Brief
4,DE-611-HS-3686373,Brief von J. V. Kotwal von Tata Institute of F...,1962-09-12,"Kotwal, J. V.; Tata Institute of Fundamental R...","Heisenberg, Werner (1901-1976)","Vaidya, Shreeniwas Maheshwar (1934-)",Brief
6,DE-611-HS-3686381,Brief von Werner Heisenberg an Deutscher Akade...,1961-01-19,"Heisenberg, Werner (1901-1976)",Deutscher Akademischer Austauschdienst (1931-1...,"Vaidya, Shreeniwas Maheshwar (1934-); Symanzik...",Brief
...,...,...,...,...,...,...,...
1995,DE-611-HS-3626349,Brief von Annemarie Giese von Max-Planck-Insti...,1966,"Giese, Annemarie; Max-Planck-Institut für Phys...",Zentralstelle für Arbeitsvermittlung (-2007),"Hermann, Klaus-Otto",Brief
1996,DE-611-HS-3626351,Brief von Zentralstelle für Arbeitsvermittlung...,1966-03-18,Zentralstelle für Arbeitsvermittlung (-2007),"Heisenberg, Werner (1901-1976); Max-Planck-Ins...","Hermann, Klaus-Otto",Brief
1997,DE-611-HS-3626361,Brief von Zentralstelle für Atomkernenergie-Do...,1968-08-14,Zentralstelle für Atomkernenergie-Dokumentatio...,"Heisenberg, Werner (1901-1976); Max-Planck-Ins...",,Brief
1998,DE-611-HS-3626366,Brief von Zentralstelle für Atomkernenergie-Do...,1968-03-25,Zentralstelle für Atomkernenergie-Dokumentatio...,"Heisenberg, Werner (1901-1976); Max-Planck-Ins...",,Brief


In [20]:
df_B.to_csv("heisenberg_namessplit_lettersonly_upscale.csv", index=False)

In [17]:
# next steps to clean data: pick columns date, senders, receivers from df and think
# about how to deal with separators in order to cleary separate the columns
#df_bibsonomy_Europa_selection = df_bibsonomy_Europa_publications[
#   ["type", "id", "tags", "label", "user", "description", "date", "authors", "publisher", "isbn"]] 


df_l_selec = df_B[["date","senders", "receivers"]]
df_l_selec

,date,senders,receivers
0,1969-01-14,"Heisenberg, Werner (1901-1976)","Votruba, Václav (1909-1990)"
1,1968-12-31,"Votruba, Václav (1909-1990)","Heisenberg, Werner (1901-1976)"
3,1962-09-18,"Heisenberg, Werner (1901-1976)","Kotwal, J. V.; Tata Institute of Fundamental R..."
4,1962-09-12,"Kotwal, J. V.; Tata Institute of Fundamental R...","Heisenberg, Werner (1901-1976)"
6,1961-01-19,"Heisenberg, Werner (1901-1976)",Deutscher Akademischer Austauschdienst (1931-1...
...,...,...,...
1995,1966,"Giese, Annemarie; Max-Planck-Institut für Phys...",Zentralstelle für Arbeitsvermittlung (-2007)
1996,1966-03-18,Zentralstelle für Arbeitsvermittlung (-2007),"Heisenberg, Werner (1901-1976); Max-Planck-Ins..."
1997,1968-08-14,Zentralstelle für Atomkernenergie-Dokumentatio...,"Heisenberg, Werner (1901-1976); Max-Planck-Ins..."
1998,1968-03-25,Zentralstelle für Atomkernenergie-Dokumentatio...,"Heisenberg, Werner (1901-1976); Max-Planck-Ins..."


In [21]:
df_l_selec.to_csv("heisenberg_namesdates_upscale.csv", index=False, sep=";")